# 라우터 MoAI-Embedding-0.6B 파인튜닝 (Colab)

**router_finetune_record.md를 재현하는 노트북입니다.** 그 문서는 이미 `train_router_moai_embedding.ipynb`가 있다고 언급하지만 그 실물 코드는 없고, 문서에 적힌 수치·설정만 존재합니다. 이 노트북은 그 문서(§2~§8)의 서술로부터 코드를 다시 구성한 것이며, 여기서는 학습셋만 레코드의 원본(`testset_natural_450` 단독)에서 **`router_trainset_v2_natural450_merged.jsonl`(v2+450 병합, 2,115건)로 교체**했습니다.

## 레코드에서 그대로 가져온 것

1. **모델**: `BCCard/MoAI-Embedding-0.6B` (§2). Hugging Face에서 실물 확인 — Qwen3-0.6B 기반 임베딩 특화 모델(hidden=1024, layer=28), `config.json`의 `pad_token_id`가 비어 있어 **레코드가 지적한 대로 수동 세팅이 실제로 필요**합니다(§2 마지막 문단). `AutoModelForSequenceClassification`으로 로드하면 `architectures: ["Qwen3Model"]`을 무시하고 `Qwen3ForSequenceClassification`으로 결합되며 `score` 헤드만 신규 초기화됩니다.
2. **학습 상수** (§4 "MoAI 학습 상수" 확정 행): `LR 1e-5` · `batch 4 × grad_accum 4`(유효배치 16, kf와 LR 의미 일치) · `bf16/fp16 자동` · `gradient checkpointing` · **full-FT**(LoRA 아님 — §4 "0.6B full-FT × 450건에 2e-5는 붕괴 위험"이라는 문구에서 명시).
3. **환경 고정** (§4, §8): `transformers==4.57.6` · `accelerate>=1.1.0`. 이 조합 자체가 `numpy.rec` 커널 오류의 원인(accelerate 0.33의 `numpy<2` 강제)을 없애므로, ELECTRA 노트북에 있던 강제 커널 재시작(`os.kill`) 셀이 **필요 없습니다**.
4. **warmup 0.1** (§4): 분류 헤드가 랜덤 초기화라 warmup 없이 최대 LR로 진입하던 문제의 확정 수정.
5. **데이터 역할** (§3-1, §3-2-1/2): 학습셋은 **분할 없이 전량 사용**(v1이 450을 360/90으로 쪼개 val macro_f1로 체크포인트를 고른 게 문제였음 — 승부처는 dev의 G5b AUROC인데 다른 축). `router_val.jsonl`을 따로 올리지 않는 한 이 노트북도 검증 분할을 하지 않습니다.
6. **체크포인트 선택** (§3-2-3): `EPOCHS`를 끝까지 돌려 에폭별 체크포인트를 전부 저장(`save_only_model=True`, `save_total_limit=None`)하고, dev(`TEST_PATH`, 1150)로 사후 선택합니다. `EPOCHS=8`은 레코드의 채택 체크포인트 스텝(kf 203 · MoAI 232)을 8로 나누면 각각 ~25.4 · ~29 스텝/에폭으로 450건·유효배치16 학습과 맞아떨어져 역산한 값입니다(레코드에 EPOCHS가 직접 적혀 있진 않음 — 이 노트북에서 **추정**했다는 점을 명시합니다).
7. **스코어 = Energy 채택** (§3-3, §5): MoAI는 MSP가 아니라 **Energy(−logsumexp)**가 우세 스코어로 확정됐습니다. 체크포인트 선택과 최종 평가 모두 Energy를 기준으로 하되, 비교를 위해 MSP도 함께 출력합니다(재학습 비용 0).
8. **G6(210건) 제외** (§3-3): 설계상 임계 보정 전용 버킷이라 AUROC 계산 시 양성·음성 풀 모두에서 제외합니다.
9. **bf16 자동판정으로 §7의 한계 보완**: 레코드 §7은 "MoAI 학습의 bf16 표기가 T4에서 비가속(에뮬레이션)일 가능성"을 한계로 남겼습니다. 이 노트북은 `torch.cuda.is_bf16_supported()`로 실제 하드웨어 지원 여부를 런타임에 판정해 T4에서는 자동으로 fp16을 쓰도록 해 그 문제를 원천적으로 피합니다.

## 레코드에 없어서 추정한 값 (주의)

`MAX_LEN=64`(§4에 명시 없음, 같은 도메인 "민원체 단문" 기준으로 ELECTRA/kf 계열과 통일), `WEIGHT_DECAY=0.01`(§4에 명시 없음, kf/ELECTRA 기본값 차용), `INFER_BATCH_SIZE=32`(평가용 배치 추론 크기, 레코드는 배치 추론 방식을 명시하지 않음). 이 값들은 근거가 약하므로 실제 팀 설정과 다르면 셀 3에서 바꾸세요.

## 학습셋 교체로 인해 달라지는 것

레코드 §5의 수치(G5b AUROC 0.830 등)는 **`testset_natural_450` 단독 학습** 결과입니다. 이 노트북은 `router_trainset_v2_natural450_merged.jsonl`(2,115건)로 학습하므로 나온 수치는 레코드 §5와 **직접 비교 불가**합니다(학습 데이터가 다름). 레코드 수치를 그대로 재현하려면 셀 3의 `TRAIN_PATH`를 `testset_natural_450.jsonl`로 바꾸고 `VAL_PATH`는 비워둔 채 재실행하세요.


In [ ]:
# 1. 환경 설정 — router_finetune_record.md §4/§8에서 확정된 조합
#    (accelerate>=1.1.0이 0.33의 numpy<2 강제를 없애므로 numpy.rec 커널 재시작 훅이 필요 없다)
!pip install -q -U transformers==4.57.6 "accelerate>=1.1.0" scikit-learn


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# 스키마 흡수 헬퍼 — 학습셋(question/business_function)과 dev 1150(text/domain) 모두 처리
def get_text(r: dict) -> str:
    return r.get("text") or r.get("question")


def get_label(r: dict):
    return r.get("business_function") or r.get("domain")


def is_oos(r: dict) -> bool:
    return bool(r.get("is_oos", False))


def is_g6(r: dict) -> bool:
    # G6(210건)은 설계상 임계 보정 전용 — AUROC 풀에서 양성/음성 모두 제외(§3-3)
    return r.get("group") == "G6"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


In [ ]:
# 2. 설정 — router_finetune_record.md §4 "MoAI 학습 상수" 확정 행을 그대로 사용

MODEL_NAME = "BCCard/MoAI-Embedding-0.6B"  # §2 — Qwen3-0.6B 기반, 임베딩 특화 백본

# 데이터 경로 — /content/data 에 아래 파일을 올린 뒤 실행
DATA_DIR = Path("/content/data")
TRAIN_PATH = DATA_DIR / "router_trainset_v2_natural450_merged.jsonl"  # 학습(in-scope, v2+450 병합, 2,115건) — 필수
VAL_PATH = DATA_DIR / "router_val.jsonl"                  # 기본적으로 없음 — §3-2-1 문제 재발 방지 위해 자동 분할하지 않음
TEST_PATH = DATA_DIR / "testset_router_oos_1150.jsonl"    # dev — 체크포인트 선택 + Energy/MSP AUROC용(1150, text+is_oos+group)

# 레코드 §4 확정값
LR = 1e-5
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 4          # 유효배치 4*4=16 — kf와 LR 의미 일치(§4)
WARMUP_RATIO = 0.1
EPOCHS = 8                    # §3-2-3 "학습 1회=EPOCHS 스윕 8포인트" + 채택 ckpt 스텝(kf 203/MoAI 232)에서 역산(추정치)

# 레코드에 값이 없어 추정한 항목 — 근거 약함, 필요시 조정
MAX_LEN = 64
WEIGHT_DECAY = 0.01
INFER_BATCH_SIZE = 32

SEED = 42             # 다중시드 보고 시 42,1,7,100,2024 등으로 반복(§6-4-1/§8-3)

OUTPUT_DIR = Path("/content/router_moai_out_v2trainset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print("model:", MODEL_NAME)


In [ ]:
# 3. 데이터 로딩 — 학습셋은 분할 없이 전량 사용(§3-1 "학습 — 전량, 분할 없음")


def load_jsonl(path: Path) -> list:
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_rows = load_jsonl(TRAIN_PATH)
assert train_rows, (
    f"{TRAIN_PATH}가 비어있거나 없습니다. router_trainset_v2_natural450_merged.jsonl 을 이 경로에 올린 뒤 다시 실행하세요."
)

val_rows = load_jsonl(VAL_PATH)  # 기본적으로 빈 리스트 — TRAIN에서 자동 분할하지 않음(§3-2-1 재발 방지)
test_rows = load_jsonl(TEST_PATH)  # 없으면 체크포인트 선택은 val_rows 또는 마지막 에폭으로 폴백

print(f"train={len(train_rows)}  val={len(val_rows)}  dev/test(mixed)={len(test_rows)}")


In [ ]:
# 4. 라벨 체계 — 학습 데이터의 business_function에서 자동 도출

LABELS = sorted({get_label(r) for r in train_rows if get_label(r) and not is_oos(r)})
assert LABELS, "학습 데이터에 in-scope 라벨(business_function)이 없습니다."
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}
print("LABELS:", LABELS)

train_rows = [r for r in train_rows if get_label(r) and not is_oos(r)]
val_rows = [r for r in val_rows if get_label(r) and not is_oos(r)]
print(f"학습에 실제 사용되는 in-scope 건수: train={len(train_rows)}  val={len(val_rows)}")


In [ ]:
# 5. 토크나이즈 + Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class RouterDataset(Dataset):
    def __init__(self, rows: list):
        texts = [get_text(r) for r in rows]
        self.labels = [label2id[get_label(r)] for r in rows]
        enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.labels[idx]),
        }


train_ds = RouterDataset(train_rows)
val_ds = RouterDataset(val_rows) if val_rows else None
print(f"train_ds={len(train_ds)}  val_ds={len(val_ds) if val_ds else 0}")


In [ ]:
# 6. 모델 로딩 — AutoModelForSequenceClassification이 config.model_type="qwen3"으로
#    Qwen3ForSequenceClassification을 결합하고 score 헤드만 신규 초기화한다(§2).

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

# 디코더 분류 헤드는 "pad가 아닌 가장 오른쪽 토큰" 표현을 쓴다 — pad_token_id 미설정 시 잘못된 토큰을 풀링한다(§2).
# BCCard/MoAI-Embedding-0.6B의 config.json에는 pad_token_id가 비어 있어 토크나이저 값을 직접 반영해야 한다.
assert tokenizer.pad_token_id is not None, "토크나이저에 pad_token이 없습니다 — 수동 지정 필요."
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # gradient checkpointing과 함께 쓰려면 캐시를 꺼야 한다


In [ ]:
# 7. 학습 — full-FT, 체크포인트는 macro_f1이 아니라 dev(TEST_PATH)의 Energy G5b AUROC로 사후 선택(§3-2)
#    bf16은 하드웨어를 실제로 판정해서 쓴다 — T4에서 bf16을 하드코딩하면 비가속 에뮬레이션이 된다는 게 §7의 한계였다.

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_ok = torch.cuda.is_available() and not bf16_ok
print(f"bf16={bf16_ok}  fp16={fp16_ok}")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


def build_training_args(**kw):
    try:
        return TrainingArguments(eval_strategy="epoch" if val_ds else "no", **kw)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch" if val_ds else "no", **kw)


training_args = build_training_args(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    per_device_eval_batch_size=PER_DEVICE_TRAIN_BATCH,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    bf16=bf16_ok,
    fp16=fp16_ok,
    gradient_checkpointing=True,     # §4 확정 항목 — Trainer가 내부에서 model.gradient_checkpointing_enable() 호출
    save_strategy="epoch",
    save_total_limit=None,           # 에폭별 체크포인트 전부 보존(§3-2-3)
    save_only_model=True,            # 재개 학습 안 함 — optimizer state 생략해 용량 절약
    load_best_model_at_end=False,    # macro_f1 기준 자동 선택을 끔(§3-2-1 재발 방지)
    logging_steps=10,
    seed=SEED,
    report_to="none",
    save_safetensors=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics if val_ds else None,
)

train_result = trainer.train()
if val_ds:
    print(trainer.evaluate())


In [ ]:
# 7-1. 체크포인트 사후 선택 — dev(TEST_PATH, 1150)의 Energy G5b AUROC 기준(§3-2-3, §3-3, §5)
#      G6은 양성/음성 풀 모두에서 제외(§3-3). dev가 없으면 val macro_f1로, 그마저 없으면 마지막 에폭으로 폴백.


@torch.no_grad()
def predict_scores(m, rows: list, batch_size: int = INFER_BATCH_SIZE):
    """각 문항에 대해 (argmax index, MSP, Energy=-logsumexp) 배치 추론."""
    m.eval()
    idx_all, msp_all, energy_all = [], [], []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        texts = [get_text(r) for r in chunk]
        enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        logits = m(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        idx_all.append(probs.argmax(dim=-1).cpu().numpy())
        msp_all.append(probs.max(dim=-1).values.cpu().numpy())
        energy_all.append((-torch.logsumexp(logits, dim=-1)).cpu().numpy())
    return np.concatenate(idx_all), np.concatenate(msp_all), np.concatenate(energy_all)


def g5b_energy_auroc(m, rows: list):
    rows = [r for r in rows if not is_g6(r)]
    if not rows:
        return None
    _, _, energy = predict_scores(m, rows)
    oos = np.array([is_oos(r) for r in rows])
    neg = ~oos
    pos_g5b = np.array([is_oos(r) and r.get("group") == "G5b" for r in rows])
    mask = pos_g5b | neg
    if pos_g5b.sum() >= 1 and neg.sum() >= 1:
        return "G5b_energy_auroc", roc_auc_score(oos[mask], energy[mask])
    if oos.any() and (~oos).any():
        return "overall_energy_auroc", roc_auc_score(oos, energy)
    return None


def score_checkpoint(ckpt_dir: Path) -> dict:
    m = AutoModelForSequenceClassification.from_pretrained(ckpt_dir).to(DEVICE)
    m.config.pad_token_id = tokenizer.pad_token_id
    result = {"checkpoint": ckpt_dir.name}
    try:
        scored = g5b_energy_auroc(m, test_rows) if test_rows else None
        if scored is not None:
            result["metric"], result["value"] = scored
        elif val_rows:
            result["metric"] = "val_macro_f1_fallback"
            idx, _, _ = predict_scores(m, val_rows)
            labels = np.array([label2id[get_label(r)] for r in val_rows])
            result["value"] = f1_score(labels, idx, average="macro")
        else:
            result["metric"] = "epoch_order_fallback"
            result["value"] = int(ckpt_dir.name.split("-")[-1])
    finally:
        del m
        torch.cuda.empty_cache()
    return result


ckpt_dirs = sorted(
    (OUTPUT_DIR / "checkpoints").glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)
assert ckpt_dirs, f"{OUTPUT_DIR / 'checkpoints'} 아래에 체크포인트가 없습니다."

if not test_rows:
    print("경고: TEST_PATH(dev, 1150)가 비어 있어 체크포인트 선택 근거가 약해집니다"
          "(val macro_f1 또는 마지막 에폭으로 폴백) — 가능하면 TEST_PATH를 채운 뒤 재실행하세요.")

scores = [score_checkpoint(p) for p in ckpt_dirs]
for s in scores:
    print(f"  {s['checkpoint']:>16s}  {s['metric']:<24s}  {s['value']:.4f}")

best = max(scores, key=lambda s: s["value"])
print(f"\n선택된 체크포인트: {best['checkpoint']}  ({best['metric']}={best['value']:.4f})")

model = AutoModelForSequenceClassification.from_pretrained(
    OUTPUT_DIR / "checkpoints" / best["checkpoint"]
).to(DEVICE)
model.config.pad_token_id = tokenizer.pad_token_id


In [ ]:
# 8. 최종 평가 — 선택된 체크포인트 기준. Energy가 채택 스코어(§5), MSP는 비교용 병행 출력(§3-3)
#    G6은 양성/음성 풀 모두에서 제외(§3-3). 도메인 정확도는 라벨 체계가 같을 때만 계산(D코드 매핑 미확정 — §6-4-2 대기)


def evaluate_router(rows: list) -> None:
    rows = [r for r in rows if not is_g6(r)]
    if not rows:
        print("dev(G6 제외)가 비어 있어 평가를 건너뜁니다.")
        return

    pred_idx, msp, energy = predict_scores(model, rows)
    oos = np.array([is_oos(r) for r in rows])

    # (a) in-scope 6-class 정확도 — 평가 라벨이 학습 LABELS와 같을 때만
    in_scope_rows = [(i, r) for i, r in enumerate(rows)
                     if not is_oos(r) and get_label(r) in label2id]
    if in_scope_rows:
        idxs, rows_ = zip(*in_scope_rows)
        y_true = [label2id[get_label(r)] for r in rows_]
        y_pred = [pred_idx[i] for i in idxs]
        print(f"[in-scope] n={len(y_true)}  accuracy={accuracy_score(y_true, y_pred):.3f}  "
              f"macro_f1={f1_score(y_true, y_pred, average='macro'):.3f}")
        print(confusion_matrix(y_true, y_pred))
    else:
        print("[in-scope] 도메인 정확도 skip — 학습 라벨(업무명)과 평가 라벨(D코드) 체계가 다름(§6-4-2 대기).")

    # (b) OOS 탐지 — Energy(채택) + MSP(비교) 병행
    if oos.any() and (~oos).any():
        neg = ~oos
        for name, score in (("Energy", energy), ("MSP(1-msp)", 1 - msp)):
            print(f"[OOS 탐지 · {name}] 전체 AUROC={roc_auc_score(oos, score):.3f}")
            for g in sorted({r.get("group") for r in rows if is_oos(r)}):
                pos = np.array([is_oos(r) and r.get("group") == g for r in rows])
                m = pos | neg
                if pos.sum() >= 1 and neg.sum() >= 1:
                    tag = " <- near-OOD 승부처" if g == "G5b" else (" (far-OOD)" if g == "G5a" else "")
                    print(f"   {g} AUROC={roc_auc_score(oos[m], score[m]):.3f}  (n_oos={int(pos.sum())}){tag}")
    else:
        print("dev에 in-scope/OOS가 둘 다 있어야 AUROC를 계산할 수 있습니다.")


evaluate_router(test_rows)


In [ ]:
# 9. 모델 저장 (+ 코랩 밖으로 내보내기용 zip)

final_dir = OUTPUT_DIR / "final"
model.save_pretrained(final_dir, safe_serialization=False)
tokenizer.save_pretrained(final_dir)

import shutil
zip_path = shutil.make_archive(str(OUTPUT_DIR / "router_moai_embedding_model_v2trainset"), "zip", final_dir)
print("saved:", zip_path)

# 구글 드라이브에 보관하려면:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(zip_path, '/content/drive/MyDrive/')


## 다음에 할 일

- **레코드 §5와 직접 비교하려면**: 셀 3의 `TRAIN_PATH`를 `testset_natural_450.jsonl`로, `VAL_PATH`는 빈 상태로 두고 재실행하면 원본과 같은 학습 조건이 됩니다. 지금 이 노트북(병합 2,115건)의 결과는 그와 다른 실험입니다.
- **§6-4 확정 조건 잔여분**:
  1. 다중 시드(예: 42, 1, 7, 100, 2024) 평균±표준편차 — 셀 2의 `SEED`를 바꿔가며 재실행 후 집계.
  2. D코드↔업무 라벨 매핑 확정 → in-scope 정확도 가드레일 활성화.
  3. Energy 스케일에서 BLOCK/REVIEW 임계(tau) G6로 재보정(기존 MSP 기반 tau 무효).
  4. 신규 200 test로 최종 1회 검증.
- **kf-deberta 대응 노트북**: 이 노트북과 같은 방식(§3-2/§3-3 반영, 학습셋만 병합본으로 교체)으로 kf-deberta-base용도 필요하면 별도로 만들어 best-vs-best 비교를 이어갈 수 있습니다.
- **실행 전**: `DATA_DIR`(`/content/data`)에 `router_trainset_v2_natural450_merged.jsonl`(필수)·`testset_router_oos_1150.jsonl`(선택, 체크포인트 선택·AUROC용 — 없으면 val macro_f1 또는 마지막 에폭으로 자동 폴백)을 올려두면 나머지 셀은 수정 없이 그대로 돌아갑니다.
